# KLEOS Logos v0.0.2 — evaluation (H9)Evaluates the trained adapter (`arm2_finetuned`) on the sealed benchmark (sha256`a11ffad7…`): greedy decoding, seed 42, `max_new_tokens` 1024. Each answer'sthinking is split off at `[/THINK]` and stored beside it; only the answer isgraded.**Before you run:**1. Settings → Accelerator → **GPU T4 x2**; **Internet on**.2. Add Input → the `kleos-policy-v0.0.7` dataset, and the **training notebook's   output** (the version that finished).3. Set `COMMIT` to the same sha the training run used.4. **Save Version → Save & Run All (Commit).** ESTIMATED 7–8 hours.**To resume**, also add this notebook's previous version output: its partial fileis copied back and finished generations are replayed, not regenerated.Then download `outputs/kleos-v007-ministral314breasoning-run1/arm2_finetuned.json` and run the H9comparison on your own machine (docs/logos.md, "H9 comparison").

In [ ]:
import timeSESSION_START = time.time()# Set COMMIT to the full sha of the pushed Kleos-Models commit that pre-registered# H9. A branch name is refused: the run must be reproducible from its manifest.COMMIT = "SET-ME"REPO_URL = "https://github.com/TejasNaik24/Kleos-Models.git"REPO = "/tmp/kleos-models"  # code stays out of the saved outputEXPERIMENT_ID = "kleos-v007-ministral314breasoning-run1"DATASET = "/tmp/kleos-data/kleos-policy-v0.0.7"OUTPUTS = "/kaggle/working/outputs"CONFIG = "configs/training/kleos_logos_v002.yaml"CONFIG_HASH = "d1961583546b5761dd854cfe845838ca91a87ea6189d54be617de21085e8cfa9"TEST_SHA256 = "a4decaaf029b227366846d44018b0b1b669d21f068800a9f3175e15ec4783980"BENCHMARK_SHA256 = "a11ffad75f5147f9d0ddad7bad4bfc073dc730df2b19173ff642774233b4b266"if len(COMMIT) != 40 or any(c not in "0123456789abcdef" for c in COMMIT):    raise SystemExit("Set COMMIT to the 40-character sha of the pushed commit (not 'main').")

In [ ]:
import globimport hashlibimport jsonimport osimport shutilimport subprocessimport sysimport threadingimport timePY = sys.executable# Kaggle ends a GPU session at 12 hours; long commands stop here instead, so this# version still finishes and saves its output for the next one to resume from.SESSION_DEADLINE = SESSION_START + 11.25 * 3600STOPPED = Falsedef run(*args, cwd=None, deadline=False):    """Run a command, streaming its output, and stop the notebook if it fails.    With deadline=True the command is stopped at SESSION_DEADLINE; every later    run() is then skipped, and the version ends normally.    """    global STOPPED    command = " ".join(str(a) for a in args)    if STOPPED:        print("skipped (session deadline reached):", command, flush=True)        return ""    print("$", command, flush=True)    process = subprocess.Popen(        [str(a) for a in args],        cwd=cwd or REPO,        stdout=subprocess.PIPE,        stderr=subprocess.STDOUT,        text=True,    )    lines = []    def pump():        for line in process.stdout:            print(line, end="", flush=True)            lines.append(line)    reader = threading.Thread(target=pump, daemon=True)    reader.start()    while process.poll() is None:        if deadline and time.time() > SESSION_DEADLINE:            print("SESSION DEADLINE: stopping so this version's output is saved. "                  "Attach it as input and Save & Run All again to resume.", flush=True)            process.terminate()            try:                process.wait(timeout=120)            except subprocess.TimeoutExpired:                process.kill()                process.wait()            reader.join(timeout=10)            STOPPED = True            return "".join(lines)        time.sleep(2)    reader.join()    if process.returncode != 0:        raise SystemExit(f"FAILED (exit {process.returncode}): {command}")    return "".join(lines)def sha256(path):    with open(path, "rb") as handle:        return hashlib.sha256(handle.read()).hexdigest()def copy_writable(source, target):    """Copy a tree out of /kaggle/input (read-only) so it can be written and removed."""    shutil.copytree(source, target, copy_function=shutil.copyfile)    for root, _, files in os.walk(target):        os.chmod(root, 0o755)        for name in files:            os.chmod(os.path.join(root, name), 0o644)

## Hardware, code, libraries, data

In [ ]:
# Two T4s (Settings -> Accelerator -> GPU T4 x2) and room for the 28 GB checkpoint.gpus = subprocess.check_output(    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip().splitlines()print("\n".join(gpus))if len(gpus) != 2 or not all("T4" in g for g in gpus):    raise SystemExit("Need exactly 2 x Tesla T4: Settings -> Accelerator -> GPU T4 x2.")for path in ("/kaggle/working", "/tmp"):    print(f"{path}: {shutil.disk_usage(path).free / 1024**3:.1f} GB free")if shutil.disk_usage("/tmp").free < 30 * 1024**3:    raise SystemExit("/tmp needs 30 GB free for the model weights (HF_HOME).")

In [ ]:
shutil.rmtree(REPO, ignore_errors=True)run("git", "clone", "--quiet", REPO_URL, REPO, cwd="/tmp")run("git", "checkout", "--quiet", COMMIT)print(run("git", "log", "-1", "--format=%H %s").strip())

In [ ]:
# Keeps Kaggle's torch (reinstalling it breaks CUDA), then pins the libraries# Hermes and Logos v0.0.1 ran with. --no-deps stops pip replacing torch.run(PY, "scripts/colab_setup.py")run(PY, "-m", "pip", "install", "-q", "--no-deps", *"transformers==5.16.1 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.50.2 tokenizers==0.23.2".split())

In [ ]:
# Weights on /tmp, never /kaggle/working (saved output, 20 GB). No KLEOS_*# variables: they change config_hash.os.environ["HF_HOME"] = "/tmp/hf_cache"os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"# Line-by-line logs from child processes, kept even if one is stopped.os.environ["PYTHONUNBUFFERED"] = "1"os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"stray = [k for k in os.environ if k.startswith("KLEOS_")]if stray:    raise SystemExit(f"Unset {stray}: KLEOS_* variables change config_hash.")run(PY, "-c", "import torch, transformers, peft, bitsandbytes; "    "print(torch.__version__, torch.cuda.device_count(), transformers.__version__, "    "peft.__version__, bitsandbytes.__version__)")

In [ ]:
# The private Kaggle dataset holding releases/kleos-policy-v0.0.7 (Add Input),# copied to the path the config hash was computed with, in /tmp, which Kaggle# never saves. Identical copies count once; two different releases stop the run.RELEASE_FILES = ("train.jsonl", "validation.jsonl", "test.jsonl", "manifest.json", "RELEASE.lock")releases = {}for path in sorted(glob.glob("/kaggle/input/**/RELEASE.lock", recursive=True)):    directory = os.path.dirname(path)    if all(os.path.exists(os.path.join(directory, n)) for n in RELEASE_FILES):        digest = tuple(sha256(os.path.join(directory, n)) for n in RELEASE_FILES)        releases.setdefault(digest, directory)if len(releases) != 1:    raise SystemExit(f"Attach exactly one kleos-policy-v0.0.7 release; found {sorted(releases.values())}.")source = next(iter(releases.values()))shutil.rmtree(DATASET, ignore_errors=True)copy_writable(source, DATASET)if sha256(os.path.join(DATASET, "test.jsonl")) != TEST_SHA256:    raise SystemExit("test.jsonl is not the sealed one (sha256 a4decaaf...).")print("dataset:", source, "->", DATASET)

## The adapter, the benchmark, any partial run

In [ ]:
adapters = sorted(    os.path.dirname(p)    for p in glob.glob(f"/kaggle/input/**/outputs/{EXPERIMENT_ID}/adapter/adapter_config.json",                       recursive=True))if len(adapters) != 1:    raise SystemExit(f"Attach exactly one finished training output; found {adapters}.")ADAPTER = adapters[0]RESULT = os.path.join(OUTPUTS, EXPERIMENT_ID, "arm2_finetuned.json")os.makedirs(os.path.dirname(RESULT), exist_ok=True)partials = glob.glob(f"/kaggle/input/**/outputs/{EXPERIMENT_ID}/arm2_finetuned.json.partial.jsonl",                     recursive=True)if len(partials) > 1:    raise SystemExit(f"Attach at most one previous evaluation version; found {partials}.")if partials and not os.path.exists(RESULT + ".partial.jsonl"):    shutil.copyfile(partials[0], RESULT + ".partial.jsonl")print("adapter:", ADAPTER, "| partial restored:", bool(partials))run(PY, "scripts/build_benchmark.py", "--dataset", DATASET, "--output", "/tmp/benchmark")if sha256("/tmp/benchmark/benchmark.jsonl") != BENCHMARK_SHA256:    raise SystemExit("The benchmark is not the sealed one (sha256 a11ffad7...).")

## Evaluate arm2 (re-running this cell resumes)

In [ ]:
run(PY, "scripts/evaluate.py", "--config", CONFIG, "--arm", "arm2_finetuned",    "--adapter", ADAPTER, "--benchmark", "/tmp/benchmark/benchmark.jsonl",    "--output", RESULT, "--resume", deadline=True)

In [ ]:
# A first look. H9 is decided only by compare.py against Hermes (docs/logos.md).if not os.path.exists(RESULT):    print("NOT FINISHED in this session. Add this version's output as an input and "          "Save & Run All again: finished generations are replayed, not regenerated.")else:    with open(RESULT) as handle:        result = json.load(handle)    print("overall:", result.get("overall"))    print("answerable:", ((result.get("corrected") or {}).get("subsets") or {}).get("answerable"))    stats = result.get("generation_stats") or {}    for key in ("responses", "thinking_truncated", "hit_max_new_tokens", "parse_failures",                "completion_tokens", "reasoning_responses", "reasoning_chars", "latency_seconds"):        print(f"{key}: {stats.get(key)}")